# 07 — Agentic RAG: the pipeline becomes a loop with a brain

Companion notebook to blog post **07 (Agentic RAG)** — the finale. Built with the
[OpenAI Agents SDK](https://github.com/openai/openai-agents-python); further reading
is listed in the series' `resources.md`.

The researcher metaphor (the librarian of post 03, promoted):
1. **Thinks before fetching** — need a search? which tool? what should the query SAY?
2. **Judges what came back** — good enough, or search again differently?
3. **Uses many sources** — handbook, bookings database, keyword index
4. **Costs more than a librarian** — hire one when the question deserves it

Agent behavior varies between runs — tool-call wording and order will differ; the
DECISIONS (routing, self-translation, multi-hop) are what reproduce. Needs an OpenAI
API key.

In [ ]:
%pip install -q openai-agents fastembed numpy

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## The tool belt — the whole series, as ingredients

Post 2b's vector search, post 2a's BM25, and a tiny bookings "database" (the classic
SQL source, minimized to a dict).

In [ ]:
import math
import numpy as np
from collections import Counter
from fastembed import TextEmbedding

corpus = [
    "Grooming appointments must be booked at least 48 hours in advance",           # 0
    "The boarding facility closes at 7 pm on weekdays and 5 pm on weekends",       # 1
    "Dogs staying longer than three nights receive a complimentary bath before pickup",  # 2
    "Refunds for cancelled boarding are issued within 5 business days",            # 3
    "Bookings made for public holidays are non-refundable",                        # 4
    "Refunds for cancelled grooming appointments are issued within 10 business days",    # 5
    "All pets must have up to date rabies vaccination records on file",            # 6
    "Daycare drop off starts at 6:30 am and the last pickup is at 8 pm",           # 7
    "A late pickup fee of 15 dollars applies for every 30 minutes after closing",  # 8
    "Error E-4042 refund transaction declined by the payment gateway",             # 9
    "Error E-4043 refund transaction succeeded but receipt email failed",          # 10
    "Error E-4044 refund transaction pending manual review",                       # 11
]

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
doc_embs = list(emb_model.embed(corpus))

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# post 2a's BM25, condensed
def tokenize(t): return t.lower().split()
docs_tok = [tokenize(d) for d in corpus]
N = len(docs_tok); avgdl = sum(len(d) for d in docs_tok) / N
df = Counter()
for d in docs_tok:
    for term in set(d):
        df[term] += 1

def bm25_score(query, doc):
    freqs = Counter(doc); score = 0.0
    for t in tokenize(query):
        if t in freqs:
            f, dl = freqs[t], len(doc)
            idf = math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5))
            score += idf * f * 2.5 / (f + 1.5 * (1 - 0.75 + 0.75 * dl / avgdl))
    return score

BOOKINGS = {
    "B-1001": {"customer": "Maya", "service": "boarding", "nights": 5, "public_holiday": False},
    "B-1002": {"customer": "Tom",  "service": "boarding", "nights": 2, "public_holiday": True},
    "B-1003": {"customer": "Ana",  "service": "grooming", "nights": 0, "public_holiday": False},
}

## The hands — three tools

A tool = a Python function with a good docstring. The docstring is not documentation —
it's the ROUTING TABLE: it's what the agent reads when choosing a tool. Note the
meaning-search docstring teaching post 04's dialect lesson ("prefer weekday/weekend").

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def search_handbook(query: str) -> str:
    """Search the pet care handbook BY MEANING. Best for policy questions
    (hours, refunds, fees, requirements). Handbooks use general policy language:
    prefer category words like weekday/weekend, boarding/daycare over specific
    days or pet names."""
    q = list(emb_model.embed([query]))[0]
    top = sorted(((cosine(q, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:3]
    return "\n".join(f"(score {s:.2f}) {corpus[i]}" for s, i in top)

@function_tool
def keyword_search(query: str) -> str:
    """Search the handbook by EXACT keyword match. Best for identifiers:
    error codes, form numbers, product names — anything where the literal
    token must match."""
    scored = sorted(((bm25_score(query, d), i) for i, d in enumerate(docs_tok)), reverse=True)
    hits = [(s, i) for s, i in scored if s > 0][:3]
    return "\n".join(f"(score {s:.2f}) {corpus[i]}" for s, i in hits) or "No keyword matches."

@function_tool
def get_booking(booking_id: str) -> str:
    """Look up a customer booking by its id (e.g. 'B-1001'). Returns the
    service type, number of nights, and whether it falls on a public holiday."""
    b = BOOKINGS.get(booking_id.upper())
    return str(b) if b else f"No booking found with id {booking_id}."

## The brain — the agent

Instructions encode the researcher's rules: decide-before-fetching, judge-and-retry
(Self-RAG), many sources — and post 00's grounding discipline survives in the last line.

In [ ]:
agent = Agent(
    name="Sunnyvale Support Agent",
    model="gpt-5.4-mini",
    instructions=(
        "You are a support agent for Sunnyvale Pet Care Center. Answer customer "
        "questions using the tools.\n"
        "- Decide whether you need to search at all; greetings and thanks need no tools.\n"
        "- Pick the right tool: meaning search for policies, keyword search for "
        "identifiers like error codes, booking lookup for anything about a "
        "customer's own booking.\n"
        "- After a search, JUDGE the results: do they actually answer the question? "
        "If not, search again with different, more general wording (handbooks say "
        "'weekends', not 'Saturday').\n"
        "- Multi-part questions may need several tools. Answer only from tool "
        "results; if nothing answers it, say you don't know."
    ),
    tools=[search_handbook, keyword_search, get_booking],
)

## The loop — the runtime, plus a trace printer

`Runner` runs the think→act→observe loop: send conversation to the LLM, execute the
tool it names, append the result to the scratchpad, repeat — until a plain-text answer
or `max_turns`. (In notebooks use `await Runner.run(...)` — `run_sync` can't start
inside Jupyter's event loop.)

In [ ]:
import json as _json

async def trace(question):
    print(f"Q: {question}\n" + "-" * 70)
    result = await Runner.run(agent, question, max_turns=8)
    for item in result.new_items:
        if item.type == "tool_call_item":
            raw = item.raw_item
            print(f"  -> CALL {raw.name}({raw.arguments})")
        elif item.type == "tool_call_output_item":
            out = str(item.output).replace("\n", "\n             ")
            print(f"     RESULT: {out[:220]}")
    print(f"  ANSWER: {result.final_output}")

## Trace 1 — knowing when NOT to retrieve

The fixed pipeline would have embedded "thanks" and searched the handbook with it.

In [ ]:
await trace("Thanks, that's all I needed!")

## Trace 2 — routing: the detective summoned on demand

Post 2b proved meaning-search ranks the wrong error-twin first. The agent reads the
docstrings and picks the keyword tool for an identifier — a CHOICE instead of 2c's
always-on fusion.

In [ ]:
await trace("What does error E-4042 mean?")

## Trace 3 — the Saturday question: post 04, self-serve

Watch the query the agent writes: the user says "Saturday", the agent searches
"weekends". Query transformation performed on the fly, only because this question
needed it.

In [ ]:
await trace("Until what time can I pick up my dog on a Saturday? He is boarding.")

## Trace 4 — multi-hop: two sources, one compound answer

The bath answer depends on data that isn't in the handbook (nights booked). A fixed
pipeline cannot express this question.

In [ ]:
await trace("My booking is B-1001. Does my dog get a free bath, and if I cancel, do I get a refund?")

## Trace 5 — the finale: the holiday trap

The agent reads the booking, learns `public_holiday: True`, writes that fact INTO its
own search query, retrieves rule + exception together, and applies the exception. The
series-long villain, handled by a system that thinks between retrievals.

In [ ]:
await trace("I need to cancel booking B-1002. Will I get my money back?")

## Standard vs agentic — and the bill

| Aspect | Standard RAG (00–06) | Agentic RAG |
|---|---|---|
| Flow | Fixed pipeline | Dynamic loop |
| Retrievals | One | As many as needed |
| Tools | One | Many |
| Query rewriting | Always-on (04) | Agent decides, per question |
| Judges retrieval | No | Yes (Self-RAG) |
| Multi-hop | Can't | Can |
| Latency / cost | Low | Higher — LLM call per step |
| Debugging | Deterministic | Trace-reading |

The researcher costs more than the librarian: more LLM turns, non-deterministic paths,
needs a stopping rule (`max_turns`). Use it for multi-hop / multi-source / messy
questions; keep the fixed pipeline for simple high-volume traffic — and put post 06's
semantic cache in front of the agent for the best of both.

**Patterns you just ran:** ReAct (the loop itself), Self-RAG (judge-and-retry in the
instructions), CRAG (fallback routing between tools).

## The whole series on one tool belt

2a detective → `keyword_search` · 2b map → `search_handbook` · 2c committee → the
agent's routing decision · 03 librarian → rerank any wide pool · 04 phrasebook → the
agent's own rewrites · 05 stamps → index-side prep · 06 receptionist → cache in front.

Retrieval stopped being a pipeline and became a decision. That's agentic RAG.